In [2]:
import numpy as np
from langchain_community.vectorstores import FAISS
# from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

In [3]:
# Relevant health & wellness documents plus distractors from other domains.
# Add a `domain` field so we can optionally filter retrieval to health-only.
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1", "domain": "health"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2", "domain": "health"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3", "domain": "health"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4", "domain": "health"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5", "domain": "health"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1", "domain": "other"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2", "domain": "other"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3", "domain": "other"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4", "domain": "other"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5", "domain": "other"}),
]

In [4]:
# Initialize OpenAI embeddings

embedding_model = OpenAIEmbeddings()

# Create FAISS vector store

vectorstore = FAISS.from_documents(documents=all_docs, embedding=embedding_model)

In [5]:
# Create retrievers

# Filter to health domain to avoid lexical collisions for words like "energy" and "balance".
similarity_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5, "filter": {"domain": "health"}},
)

In [6]:
query_prompt = PromptTemplate.from_template(
    """You rewrite user questions for vector retrieval.
Generate 3 alternative queries.

Rules:
- Keep the same intent and topic as the original question.
- Stay strictly in health and wellness context.
- If a term is ambiguous (e.g., energy, balance), interpret it for personal wellbeing.
- Keep each query concise.

Original question: {question}
"""
)

multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 5, "filter": {"domain": "health"}},
    ),
    llm=ChatOpenAI(model="gpt-3.5-turbo", temperature=0),
    prompt=query_prompt,
    include_original=False,
)

In [7]:
# Query

query = 'How to improve energy levels and maintain balance?'

In [8]:
# Retrieve results

similarity_results = similarity_retriever.invoke(query)

# MultiQuery returns a unique union across generated queries, often > k.
# Re-rank that union against the original query and keep top-5 for fair comparison.
multiquery_union_results = multiquery_retriever.invoke(query)

query_embedding = np.array(embedding_model.embed_query(query))
doc_texts = [doc.page_content for doc in multiquery_union_results]
doc_embeddings = embedding_model.embed_documents(doc_texts)

scored_docs = []
for doc, emb in zip(multiquery_union_results, doc_embeddings):
    emb = np.array(emb)
    score = float(np.dot(query_embedding, emb) / (np.linalg.norm(query_embedding) * np.linalg.norm(emb)))
    scored_docs.append((doc, score))

scored_docs.sort(key=lambda x: x[1], reverse=True)
multiquery_results = [doc for doc, _ in scored_docs[:5]]

print(f"Similarity count: {len(similarity_results)}")
print(f"MultiQuery union count: {len(multiquery_union_results)}")
print(f"MultiQuery re-ranked count: {len(multiquery_results)}")

In [9]:
for i, doc in enumerate(similarity_results):
    print(f"\n-- Similarity Result {i+1} --")
    print(doc.page_content)

print("*"*150)

for i, doc in enumerate(multiquery_results):
    print(f"\n-- Multiquery Result {i+1} --")
    print(doc.page_content)


-- Similarity Result 1 --
Drinking sufficient water throughout the day helps maintain metabolism and energy.

-- Similarity Result 2 --
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

-- Similarity Result 3 --
Regular walking boosts heart health and can reduce symptoms of depression.

-- Similarity Result 4 --
Deep sleep is crucial for cellular repair and emotional regulation.

-- Similarity Result 5 --
The solar energy system in modern homes helps balance electricity demand.
******************************************************************************************************************************************************

-- Multiquery Result 1 --
Drinking sufficient water throughout the day helps maintain metabolism and energy.

-- Multiquery Result 2 --
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

-- Multiquery Result 3 --
Regular walking boosts heart health and can reduce symptoms of depression.

-- Multiquery 